# 20 - עמידות דינמית - שעות שיא מול שעות שפל

כל סיפור העמידות המסופר במחברות `03`-`06` נשען על גרף יחיד, **ממוצע על פני זמן**: קשת
`u - v` קיימת אם *כלשהי* נסיעה ב*כלשהו* יום שירות עוברת מ-`u` ל-`v`, ומשקלה הוא המספר הכולל של
נסיעות כאלה. גרף זה מתאר רשת שאינה קיימת במציאות - אף נוסע אינו נוסע לעולם על הממוצע של 24
השעות. מחברה `19` חתכה את הפיד לחלונות זמן ובנתה מחדש את הגרף בנפרד בתוך כל אחד מהם.
מחברה זו משתמשת בגרפים הללו לפי חלון כדי לשאול את השאלה שהניתוח הסטטי אינו יכול לענות עליה:

> **האם הקריטיות תלויה בשעת היום?** האם אותן תחנות קריטיות בשעת השיא של הבוקר ובלילה,
> והאם הרשת שבירה יותר כאשר השירות דליל?

כאן מורצים שני ניסויים.

1. **יציבות דירוגים בין חלונות.** עבור כל חלון אנו מחשבים מחדש degree, weighted degree ו-betweenness
   מדגמי, מדרגים את התחנות, ואז משווים דירוגים *בין* חלונות באמצעות מתאם Spearman וחפיפת top-50.
   מכיוון ש-betweenness מדגמי הוא רועש, אנו גם מעריכים את **רצפת הרעש** (noise floor) - מתאם ה-Spearman
   בין שני מדגמי betweenness בלתי תלויים של *אותו* חלון - כך שניתן לשפוט אי-הסכמה בין חלונות
   ביחס לאי-ההסכמה שהאומד מייצר בכוחות עצמו.
2. **תקיפה ממוקדת מול אקראית, בנפרד לכל חלון.** אותה סימולציה של הסרה הדרגתית כמו במחברה
   `06`, אך מורצת בתוך הגרף של כל חלון, כאשר רכיב הקשירות הגדול ביותר מנורמל לפי מספר התחנות
   **ששרדו** (`|LCC| / (N - k)`) - התיקון שמחברה `06` ביססה. מכיוון שלחלונות גדלים שונים,
   רשת ההסרה מבוטאת כ**שיעור מתוך תחנות אותו חלון**, כך שעקומות הנזק ברות-השוואה.

זה עונה על סעיף המצגת *"עמידות לפי שעות שיא מול שעות שפל"* (resilience at peak vs off-peak hours).

## קלט

* `outputs/nb/19_*/tables/window_summary.csv` - שורה אחת לכל חלון זמן
  (`window, start_hour, end_hour, trips, nodes, directed_edges, avg_degree, largest_component_share`).
* `outputs/nb/19_*/graph_<window>.pkl` - הגרף הבלתי מכוון לכל חלון (מועדף), **או**
  `outputs/nb/19_*/tables/edges_<window>.csv` (`from_stop, to_stop, trip_frequency`) אם ה-pickle חסר.
* `outputs/nb/02_graph_construction/tables/nodes.csv` - שמות תחנות, קואורדינטות ואזורים, בשימוש
  לצורכי תיוג התוצאות בלבד.

**מחברות שחייבות לרוץ קודם:** `02_graph_construction` ו-`19` (בונה הגרפים לפי חלונות זמן).
מחברה זו **אינה** קוראת את הקובץ הגולמי `stop_times.txt` בנפח 816 MB; כל פענוח הזמנים של GTFS
בוצע ב-`19`.

## פלט

הכל נכתב תחת `outputs/nb/20_dynamic_resilience/`:

* `tables/dynamic_resilience.csv` - `window, strategy, removed, largest_component_share_surviving`
  (טבלת החוזה; שורה אחת לכל חלון x אסטרטגיה x רמת הסרה).
* `tables/peak_vs_offpeak_centrality.csv` - `stop_id, stop_name, window, degree, weighted_degree,
  approx_betweenness, rank_in_window` (טבלת החוזה; שורה אחת לכל תחנה בכל חלון).
* `dynamic_summary.json` - מספרי הכותרת, חלונות השיא / השפל שנבחרו, וכל קבוע שנעשה בו שימוש.
* טבלאות משלימות: `window_graph_summary.csv`, `window_rank_agreement.csv`, `rank_movers.csv`,
  `vanished_at_offpeak.csv`, `dynamic_resilience_detail.csv`, `fragility_summary.csv`,
  `betweenness_noise_floor.csv`.
* איורים: `damage_curves_by_window.png`, `damage_curves_grid.png`, `rank_agreement_heatmap.png`,
  `top_rank_movers.png`, `fragility_summary.png`.

דבר מחוץ ל-`outputs/nb/20_dynamic_resilience/` אינו נכתב. התיקיות המצוטטות בדוח `outputs/tables`,
`outputs/figures` ו-`outputs/rail` אינן נוגעות כלל.

## 1. אתחול סביבת העבודה

התא שלהלן הופך את המחברת לניתנת להרצה הן על עותק מקומי והן על Google Colab. `_ensure(...)`
מתקין באמצעות pip רק את החבילות שחסרות באמת (כך שהרצה חוזרת זולה), ו-`find_repo_root()` מטפס
מעלה מתיקיית העבודה בחיפוש אחר תיקיית ה-GTFS, ומשכפל את המאגר אל `/content` אם אנו על Colab.
לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT`. כל תא מאוחר יותר מסתמך על שלושת הנתיבים הללו, ולכן
תא זה חייב לרוץ ראשון. הוא זהה לפתיח שבכל שאר המחברות בסדרה.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות שלב וקבועים מכווננים

כל זמן הריצה של מחברת זו מרוכז בקבועים שלהלן, כך שבודק יכול להחליף עומק ניתוח במהירות במקום
אחד. שני דברים עולים זמן ממשי:

| קבוע | משמעות | עלות |
|---|---|---|
| `BETWEENNESS_SAMPLES = 250` | מספר ה-pivots ל-betweenness מדגמי, **לכל חלון** | כ-250 סריקות BFS לכל חלון; 20-60 שניות על גרף חלון בן 25k צמתים |
| `RUN_NOISE_FLOOR = True` | הרצת betweenness נוספת אחת על חלון השיא עם seed שני | +הרצת betweenness אחת |
| `REMOVAL_FRACTIONS` / `EXTRA_FRACTIONS` | רשת ההסרה, המבוטאת כ**שיעור מתחנות כל חלון** | מעבר אחד של רכיבי קשירות לכל נקודה |
| `RANDOM_TRIALS = 3` | חזרות עם seed שממוצעות עבור קו הבסיס האקראי | משלש את עלות קו הבסיס |
| `ATTACK_STRATEGIES` | אילו סדרי תקיפה ממוקדים לדמות | סריקה אחת לכל אחד |

עם `W` חלונות הסימולציה מבצעת כ-`W x (len(ATTACK_STRATEGIES) + RANDOM_TRIALS) x 24` הערכות של
רכיבי קשירות בזמן לינארי - עבור 4 חלונות, 4 אסטרטגיות ו-3 ניסויים אקראיים מדובר בכ-670 מעברים,
כלומר **2-6 דקות על מחשב נייד רגיל**, בתוספת betweenness מדגמי אחד לכל חלון. כל צעד הסרה משתמש
ב-*תצוגת* `subgraph` במקום להעתיק את הגרף, ולכן צריכת הזיכרון נותרת קבועה.

`RANK_METRIC` קובע על איזו מידת מרכזיות מתבססת עמודת החוזה `rank_in_window`. ברירת המחדל היא
`approx_betweenness` משום שזו תפיסת הקריטיות של הפרויקט (כמה מקישוריות הרשת תלויה בתחנה), אך
ניתוח ההסכמה בין חלונות מורץ עבור **כל שלוש** המידות, כך שניתן לבחון את הבחירה במקום לסמוך עליה.

`PEAK_WINDOW_OVERRIDE` / `OFFPEAK_WINDOW_OVERRIDE` מאפשרים לציין בשמם את שני החלונות המושווים
זה מול זה; כאשר הם נותרים `None`, המחברת בוחרת אותם אוטומטית (סעיף 6).

In [ ]:
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import json, pickle, random, re, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.0)

# ---------------- stage folders ----------------
STAGE = OUT / '20_dynamic_resilience'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# ---------------- tunable constants: all runtime cost lives here ----------------
BETWEENNESS_SAMPLES = 250      # pivots for sampled betweenness, per window
BETWEENNESS_SEED = 42          # seed of the main sample
BETWEENNESS_SEED_B = 7         # second seed, used only for the noise-floor check
RUN_NOISE_FLOOR = True         # False -> skip the second betweenness run

RANK_METRIC = 'approx_betweenness'   # metric behind the contract column rank_in_window
AGREEMENT_METRICS = ['degree', 'weighted_degree', 'approx_betweenness']
TOP_OVERLAP_N = 50             # the "top-50 overlap" of the agreement table
MOVER_POOL = 2000              # a station must be top-2000 in at least one window to count as a mover
TOP_MOVERS = 20                # rows in the mover figure (per direction)

# removal grid, as a SHARE of each window's own station count (windows differ in size)
REMOVAL_FRACTIONS = [round(x, 4) for x in np.linspace(0.0, 0.10, 21)]
EXTRA_FRACTIONS = [0.0005, 0.001, 0.0025]    # small-k anchors so the early curve is resolved
RANDOM_TRIALS = 3              # seeded trials averaged for the random baseline
SEED = 42
ATTACK_STRATEGIES = ['degree', 'weighted degree', 'betweenness', 'articulation points']

PEAK_WINDOW_OVERRIDE = None    # e.g. 'am_peak'  -> forces the peak side of the head-to-head
OFFPEAK_WINDOW_OVERRIDE = None # e.g. 'night'    -> forces the off-peak side

FIG_DPI = 150

assert RANK_METRIC in AGREEMENT_METRICS, 'RANK_METRIC must be one of AGREEMENT_METRICS'

# numpy 2 renamed trapz -> trapezoid; support both.
_trapz = getattr(np, 'trapezoid', None) or np.trapz

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('stage folder :', STAGE)
print('networkx', nx.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

## 3. עיבוד תוויות בעברית

שמות התחנות בפיד ה-GTFS הישראלי הם בעברית, ותרשים מזיזי הדירוג מדפיס אותם. Matplotlib אינה
מממשת את אלגוריתם ה-bidirectional של Unicode, ולכן טקסט מימין לשמאל יוצא הפוך ובלתי קריא. התא
שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים
עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני ציורה, ובוחר גופן שיש בו למעשה גליפים
עבריים (Arial ב-Windows, DejaVu Sans בכל מקום אחר). הפעולה אידמפוטנטית - הרצה חוזרת לא תערים
patches זה על גבי זה. כל שאר הטקסט במחברת הוא באנגלית, בהתאם לדרישת ההגשה.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור השלבים הקודמים

תיקיות השלבים מאותרות לפי **הקידומת הדו-ספרתית** שלהן (`OUT.glob("19*")`), ולעולם לא לפי slug
מדויק, כך שמחברת ששמה שונה עדיין מאותרת. אם שלב נדרש חסר, אנו עוצרים מיד עם שגיאה המציינת את
שם המחברת שיש להריץ, במקום לנתח בשקט גרף אחר.

נדרשים שני שלבים:

* **19** - הגרפים לפי חלון ו-`window_summary.csv`. דרישת חובה: בלעדיו אין מה להשוות.
* **02** - `nodes.csv`, המשמש אך ורק לתיוג (שם תחנה, קואורדינטות, אזור). אילו היה חסר, הניתוח
  עדיין היה נכון אך הטבלאות היו בלתי קריאות, ולכן גם הוא נדרש.

In [ ]:
def find_stage(prefix, must_run):
    """Return the outputs/nb stage folder whose name starts with `prefix` (e.g. '19')."""
    hits = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir()) if OUT.exists() else []
    if not hits:
        raise FileNotFoundError(
            'No stage folder starting with "' + prefix + '" under ' + str(OUT) + ' - '
            'run notebook ' + must_run + ' first.')
    return hits[0]


def find_file(root, *patterns):
    """First file under `root` matching one of the glob patterns, searched recursively."""
    if root is None or not Path(root).exists():
        return None
    for pat in patterns:
        hits = sorted(Path(root).rglob(pat))
        if hits:
            return hits[0]
    return None


STAGE19 = find_stage('19', '19 (time-windowed graph construction)')
STAGE02 = find_stage('02', '02_graph_construction')
print('window stage :', STAGE19)
print('nodes stage  :', STAGE02)

## 5. גילוי חלונות הזמן

מחברה 19 כותבת `graph_<window>.pkl` אחד ו-`tables/edges_<window>.csv` אחד לכל חלון, בתוספת
`window_summary.csv` המפרט את החלונות עם גבולות השעות שלהם. במקום לקבע בקוד את שמות החלונות
(דבר שהיה נשבר ברגע שמחברה 19 משנה את הגדרותיה), אנו **מגלים** אותם מהקבצים שעל הדיסק ואז
מחברים אותם ל-`window_summary.csv` לפי מפתח מנורמל (slugified), כך ש-`AM Peak`, `am_peak`
ו-`am-peak` כולם מתאימים. חלונות המופיעים בסיכום אך אין להם קובץ גרף מדווחים ומדולגים; קבצי
גרף ללא שורת סיכום עדיין מנותחים, פשוט אין להם גבולות שעות.

ה-pickle מועדף על פני ה-CSV משום שהוא כבר נושא את אובייקט הגרף המדויק שמחברה 19 בנתה; ה-CSV
הוא מנגנון הגיבוי כדי שמחברת זו תמשיך לעבוד אם ה-pickle אינו קריא בין גרסאות `networkx`.

In [ ]:
def _slug(s):
    return re.sub(r'[^a-z0-9]+', '_', str(s).strip().lower()).strip('_')


# --- files written by notebook 19 -----------------------------------------
pkl_by_slug, csv_by_slug = {}, {}
for p in sorted(STAGE19.rglob('graph_*.pkl')):
    pkl_by_slug.setdefault(_slug(p.stem[len('graph_'):]), p)
for p in sorted(STAGE19.rglob('edges_*.csv')):
    csv_by_slug.setdefault(_slug(p.stem[len('edges_'):]), p)

# --- window_summary.csv (hour boundaries, trip counts) --------------------
ws_path = find_file(STAGE19, 'window_summary.csv')
window_summary = None
if ws_path is not None:
    window_summary = pd.read_csv(ws_path, encoding='utf-8-sig')
    window_summary['_slug'] = window_summary['window'].map(_slug)
    print('window_summary.csv <-', ws_path, '(' + str(len(window_summary)), 'windows )')
else:
    print('WARNING: window_summary.csv not found under', STAGE19,
          '- window labels will be taken from the file names and hour boundaries will be blank.')

# --- assemble the window list ---------------------------------------------
order = (list(window_summary['_slug']) if window_summary is not None
         else sorted(set(pkl_by_slug) | set(csv_by_slug)))
seen, WINDOWS, missing = set(), [], []
for slug in order:
    if slug in seen:
        continue
    seen.add(slug)
    pkl, csv = pkl_by_slug.get(slug), csv_by_slug.get(slug)
    if pkl is None and csv is None:
        missing.append(slug)
        continue
    label = slug
    row = {}
    if window_summary is not None:
        hit = window_summary[window_summary['_slug'] == slug]
        if len(hit):
            row = hit.iloc[0].to_dict()
            label = str(row['window'])
    WINDOWS.append({'window': label, 'slug': slug, 'pkl': pkl, 'csv': csv,
                    'start_hour': row.get('start_hour'), 'end_hour': row.get('end_hour'),
                    'trips': row.get('trips')})
# graph files with no summary row
for slug in sorted(set(pkl_by_slug) | set(csv_by_slug)):
    if slug not in seen:
        WINDOWS.append({'window': slug, 'slug': slug, 'pkl': pkl_by_slug.get(slug),
                        'csv': csv_by_slug.get(slug), 'start_hour': None,
                        'end_hour': None, 'trips': None})

if not WINDOWS:
    raise FileNotFoundError(
        'No per-window graphs found under ' + str(STAGE19) + '. Expected graph_<window>.pkl or '
        'tables/edges_<window>.csv - run notebook 19 (time-windowed graph construction) first.')
if missing:
    print('WARNING: windows listed in window_summary.csv with no graph file (skipped):', missing)

print()
print('windows to analyse (' + str(len(WINDOWS)) + '):')
for w in WINDOWS:
    src = 'pickle' if w['pkl'] is not None else 'edges csv'
    print('  ' + str(w['window']).ljust(18), '| source:', src.ljust(9),
          '| hours:', w['start_hour'], '->', w['end_hour'])

## 6. טעינת הגרפים לפי חלון

כל גרף חלון נטען, נכפה להיות **בלתי מכוון** (אם מחברה 19 שמרה אובייקט מכוון, שני כיווני הנסיעה
מקופלים לקשת אחת ותדירויות הנסיעה שלהם מסוכמות - אותה הטלה שמחברה 02 משתמשת בה), מנוקה
מלולאות עצמיות, וממופה מחדש כך שכל מזהה צומת יהיה `str`. הצעד האחרון חשוב: מזהים היוצאים מתוך
pickle יכולים להיות מספרים שלמים בעוד שמזהים היוצאים מ-CSV הם מחרוזות, ואי-התאמה שקטה הייתה
גורמת לכל השוואה בין חלונות להחזיר חיתוך ריק.

מאפייני התחנות מגיעים מ-`nodes.csv` של שלב 02 ומוצמדים לצמתים הקיימים בחלון. הגדלים המתקבלים
לכל חלון נכתבים אל `tables/window_graph_summary.csv`; טבלה זו היא גם התוצאה המהותית הראשונה,
משום שהירידה במספר התחנות משעת השיא ללילה *היא* דלילות השירות.

In [ ]:
def to_undirected_sum(D):
    """Collapse a directed graph into an undirected one, summing both directions' weights."""
    U = nx.Graph()
    U.add_nodes_from(D.nodes(data=True))
    for u, v, data in D.edges(data=True):
        if u == v:
            continue
        w = float(data.get('weight', data.get('trip_frequency', 1)))
        if U.has_edge(u, v):
            U[u][v]['weight'] += w
        else:
            U.add_edge(u, v, weight=w)
    return U


def graph_from_edges_csv(path):
    """Rebuild the undirected, trip-frequency weighted window graph from edges_<window>.csv."""
    e = pd.read_csv(path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')
    wcol = next((c for c in ('trip_frequency', 'weight', 'trips', 'count') if c in e.columns), None)
    G = nx.Graph()
    ws = (pd.to_numeric(e[wcol], errors='coerce').fillna(1.0).tolist() if wcol else [1.0] * len(e))
    for a, b, w in zip(e['from_stop'].astype(str), e['to_stop'].astype(str), ws):
        if a == b:
            continue
        if G.has_edge(a, b):
            G[a][b]['weight'] += float(w)
        else:
            G.add_edge(a, b, weight=float(w))
    return G


def load_window_graph(rec):
    """Load one window as an undirected, str-labelled, self-loop-free graph."""
    G = None
    if rec['pkl'] is not None:
        try:
            with open(rec['pkl'], 'rb') as fh:
                G = pickle.load(fh)
        except Exception as exc:                     # version-fragile pickles -> fall back to csv
            print('  could not unpickle', rec['pkl'].name, '(' + type(exc).__name__ + ') - using edges csv')
            G = None
    if G is None:
        if rec['csv'] is None:
            raise FileNotFoundError('no usable graph file for window ' + str(rec['window']))
        G = graph_from_edges_csv(rec['csv'])
    if G.is_directed():
        G = to_undirected_sum(G)
    else:
        G = nx.Graph(G)
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    for _, _, d in G.edges(data=True):
        d['weight'] = float(d.get('weight', d.get('trip_frequency', 1)))
    return G


# --- station labels from stage 02 -----------------------------------------
nodes_path = find_file(STAGE02, 'nodes.csv')
if nodes_path is None:
    raise FileNotFoundError('nodes.csv not found under ' + str(STAGE02) +
                            ' - run notebook 02_graph_construction first.')
nodes_df = pd.read_csv(nodes_path, dtype={'stop_id': str}, encoding='utf-8-sig')
NODE_ATTR = nodes_df.set_index(nodes_df['stop_id'].astype(str))[
    ['stop_name', 'lat', 'lon', 'region']].to_dict('index')
print('station labels:', len(NODE_ATTR), 'stops  <-', nodes_path)
print()

# --- load every window -----------------------------------------------------
GRAPHS, rows = {}, []
for rec in WINDOWS:
    t0 = time.time()
    G = load_window_graph(rec)
    for n in G.nodes():
        a = NODE_ATTR.get(n)
        if a:
            G.nodes[n].update(a)
    GRAPHS[rec['window']] = G
    comps = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    n, m = G.number_of_nodes(), G.number_of_edges()
    rows.append({'window': rec['window'], 'start_hour': rec['start_hour'], 'end_hour': rec['end_hour'],
                 'trips_reported_by_nb19': rec['trips'], 'nodes': n, 'undirected_edges': m,
                 'avg_degree': round(2 * m / n, 3) if n else 0.0,
                 'components': len(comps),
                 'largest_component': comps[0] if comps else 0,
                 'largest_component_share': round(comps[0] / n, 4) if n else 0.0,
                 'total_edge_weight': float(sum(d['weight'] for _, _, d in G.edges(data=True)))})
    print('  ' + str(rec['window']).ljust(18), format(n, '>7,'), 'nodes ',
          format(m, '>7,'), 'edges  (' + str(round(time.time() - t0, 1)) + 's)')

window_graph_summary = pd.DataFrame(rows)
empty = window_graph_summary[window_graph_summary['nodes'] < 3]['window'].tolist()
if empty:
    print()
    print('WARNING: dropping windows with fewer than 3 stations (nothing to simulate):', empty)
    window_graph_summary = window_graph_summary[window_graph_summary['nodes'] >= 3].reset_index(drop=True)
window_graph_summary.to_csv(TABLES / 'window_graph_summary.csv', index=False, encoding='utf-8-sig')
WINDOW_NAMES = list(window_graph_summary['window'])
if not WINDOW_NAMES:
    raise ValueError('every window graph from notebook 19 is empty - nothing to analyse.')
print()
print(window_graph_summary.drop(columns=['trips_reported_by_nb19']).to_string(index=False))

## 7. בחירת חלון השיא וחלון השפל

רוב הניתוח מורץ על *כל* החלונות, אך ההשוואה הישירה (מזיזי דירוג, "אילו תחנות נעלמות בלילה")
זקוקה לשניים מהם בדיוק. מחברה 19 היא הבעלים של הגדרות החלונות, ולכן מחברת זו אינה רשאית להניח
דבר לגבי שמותיהם. הכלל שלהלן, לפי סדר:

1. `PEAK_WINDOW_OVERRIDE` / `OFFPEAK_WINDOW_OVERRIDE` מפורשים;
2. התאמה לפי מילות מפתח בשם (`peak`, `morning`, `rush`, `am` לצד השיא; `night`, `late`, `off`,
   `evening` לצד השפל);
3. בהיעדר שניהם, החלון **הגדול ביותר** במספר תחנות נלקח כשיא והחלון **הקטן ביותר** כשפל.

הכלל שהופעל בפועל מודפס ונרשם ב-`dynamic_summary.json`, כך שהבחירה ניתנת לביקורת במקום להיות
מוסתרת.

In [ ]:
PEAK_KEYWORDS = ['am_peak', 'morning_peak', 'peak_am', 'rush', 'peak', 'morning', 'am']
OFFPEAK_KEYWORDS = ['night', 'late', 'off_peak', 'offpeak', 'off', 'evening', 'early']


def _match_keyword(names, keywords):
    for kw in keywords:
        for nm in names:
            if kw in _slug(nm):
                return nm, 'name keyword "' + kw + '"'
    return None, None


def choose_windows(summary):
    names = list(summary['window'])
    by_size = summary.sort_values('nodes', ascending=False)['window'].tolist()

    if PEAK_WINDOW_OVERRIDE in names:
        peak, why_p = PEAK_WINDOW_OVERRIDE, 'PEAK_WINDOW_OVERRIDE'
    else:
        peak, why_p = _match_keyword(names, PEAK_KEYWORDS)
        if peak is None:
            peak, why_p = by_size[0], 'largest window by station count'

    off_names = [n for n in names if n != peak]
    if OFFPEAK_WINDOW_OVERRIDE in off_names:
        off, why_o = OFFPEAK_WINDOW_OVERRIDE, 'OFFPEAK_WINDOW_OVERRIDE'
    else:
        off, why_o = _match_keyword(off_names, OFFPEAK_KEYWORDS)
        if off is None:
            off = next((n for n in reversed(by_size) if n != peak), None)
            why_o = 'smallest window by station count'
    return peak, why_p, off, why_o


if len(WINDOW_NAMES) < 2:
    PEAK_WINDOW, OFFPEAK_WINDOW = WINDOW_NAMES[0], None
    WHY_PEAK = WHY_OFFPEAK = 'only one window available'
    print('WARNING: notebook 19 produced a single window (' + str(PEAK_WINDOW) + '). '
          'The cross-window comparisons below will be skipped; only the per-window '
          'resilience simulation is meaningful.')
else:
    PEAK_WINDOW, WHY_PEAK, OFFPEAK_WINDOW, WHY_OFFPEAK = choose_windows(window_graph_summary)

print('peak window     :', PEAK_WINDOW, ' (' + str(WHY_PEAK) + ')')
print('off-peak window :', OFFPEAK_WINDOW, ' (' + str(WHY_OFFPEAK) + ')')

## 8. מרכזיות בתוך כל חלון

עבור כל חלון אנו מחשבים שלוש מידות על הגרף של אותו חלון:

* **degree** - כמה תחנות שכנות נבדלות ניתן להגיע אליהן ישירות *בחלון זה*;
* **weighted degree (strength)** - אותו הדבר, משוקלל לפי מספר הנסיעות המשתמשות בכל מקטע, כלומר
  כמה שירות עובר בפועל דרך התחנה בשעות אלה;
* **approximate betweenness** - שיעור המסלולים הקצרים ביותר העוברים דרך התחנה, המוערך מתוך
  `BETWEENNESS_SAMPLES` pivots אקראיים (`nx.betweenness_centrality(..., k=...)`). חישוב betweenness
  מדויק על גרף בן כ-25k צמתים הוא שעות עבודה; המדגם חסר הטיה בתוחלת אך רועש ברמת התחנה
  הבודדת, וסעיף 10 מודד בדיוק כמה רועש. הוא מחושב על **רכיב הקשירות הגדול ביותר** של החלון
  בלבד - מרכזיות מבוססת מסלולים קצרים ביותר אינה מוגדרת בין רכיבים - וכל תחנה מחוצה לו מקבלת 0.

`rank_in_window` הוא מיקום התחנה תחת `RANK_METRIC`, כאשר דירוג 1 הוא המרכזי ביותר. שוויונות
נשברים באופן דטרמיניסטי לפי weighted degree, אחר כך degree, ואז `stop_id`, כך שהרצה חוזרת
מפיקה דירוגים זהים בית-בית; הדבר חשוב משום ש-betweenness מדגמי הוא אפס בדיוק עבור זנב ארוך של
תחנות, וסדר שבירת שוויון שרירותי היה יוצר "תזוזת דירוג" מזויפת בין חלונות.

תא זה כותב את טבלת החוזה `tables/peak_vs_offpeak_centrality.csv` עם העמודות המוסכמות בדיוק
(`stop_id, stop_name, window, degree, weighted_degree, approx_betweenness, rank_in_window`),
בתוספת `window_centrality_detail.csv` עשיר יותר הנושא קואורדינטות, אזור ודירוגים לכל מידה עבור
הניתוח שלהלן.

In [ ]:
def window_metrics(name, G, seed=BETWEENNESS_SEED):
    """degree / weighted degree / sampled betweenness + a deterministic rank, for one window."""
    if G.number_of_nodes() == 0:
        return pd.DataFrame(columns=['stop_id', 'window', 'degree', 'weighted_degree',
                                     'approx_betweenness', 'rank_in_window'])
    deg = dict(G.degree())
    wdeg = dict(G.degree(weight='weight'))
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    Gc = G.subgraph(comps[0])
    k = min(BETWEENNESS_SAMPLES, Gc.number_of_nodes())
    t0 = time.time()
    btw = nx.betweenness_centrality(Gc, k=k, seed=seed, normalized=True)
    took = time.time() - t0

    df = pd.DataFrame({'stop_id': [str(n) for n in G.nodes()]})
    df['window'] = name
    df['degree'] = df['stop_id'].map(deg).astype(float)
    df['weighted_degree'] = df['stop_id'].map(wdeg).astype(float)
    df['approx_betweenness'] = df['stop_id'].map(btw).fillna(0.0).astype(float)
    df['in_largest_component'] = df['stop_id'].isin(set(map(str, Gc.nodes())))
    df['stop_name'] = df['stop_id'].map(
        lambda s: str((NODE_ATTR.get(s) or {}).get('stop_name', '') or ''))
    df['lat'] = df['stop_id'].map(lambda s: (NODE_ATTR.get(s) or {}).get('lat'))
    df['lon'] = df['stop_id'].map(lambda s: (NODE_ATTR.get(s) or {}).get('lon'))
    df['region'] = df['stop_id'].map(
        lambda s: str((NODE_ATTR.get(s) or {}).get('region', '') or ''))

    # deterministic ranks: value desc, ties broken by weighted degree -> degree -> id
    for metric in AGREEMENT_METRICS:
        s = df.sort_values([metric, 'weighted_degree', 'degree', 'stop_id'],
                           ascending=[False, False, False, True])
        df.loc[s.index, 'rank_' + metric] = np.arange(1, len(s) + 1)
    df['rank_in_window'] = df['rank_' + RANK_METRIC].astype(int)
    df = df.sort_values('rank_in_window').reset_index(drop=True)
    print('  ' + str(name).ljust(18), 'betweenness on', format(Gc.number_of_nodes(), ','),
          'nodes with k=' + str(k), '->', str(round(took, 1)) + 's')
    return df


print('computing per-window centrality (sampled betweenness is the slow part) ...')
metric_frames = {name: window_metrics(name, GRAPHS[name]) for name in WINDOW_NAMES}
all_metrics = pd.concat(metric_frames.values(), ignore_index=True)

CONTRACT_COLS = ['stop_id', 'stop_name', 'window', 'degree', 'weighted_degree',
                 'approx_betweenness', 'rank_in_window']
all_metrics[CONTRACT_COLS].to_csv(TABLES / 'peak_vs_offpeak_centrality.csv',
                                  index=False, encoding='utf-8-sig')
all_metrics.to_csv(TABLES / 'window_centrality_detail.csv', index=False, encoding='utf-8-sig')
print()
print('wrote', TABLES / 'peak_vs_offpeak_centrality.csv', '(' + format(len(all_metrics), ','), 'rows )')
print()
print('top 10 stations by ' + RANK_METRIC + ' in each window:')
for name in WINDOW_NAMES:
    top = metric_frames[name].head(10)['stop_name'].tolist()
    print('  ' + str(name).ljust(18) + ' | ' + ' | '.join(str(t)[:22] for t in top[:5]))

## 9. האם הדירוגים מסכימים בין חלונות?

זהו לב השאלה "האם הקריטיות תלויה בשעת היום". עבור כל זוג סדור של חלונות ולכל מידה אנו מדווחים:

* **`spearman_rho`** - מתאם הדירוגים, המחושב **רק על תחנות הקיימות בשני החלונות**. הוא ממומש
  ישירות מדירוגים ממוצעים בתוספת מתאם Pearson, כך שלא מוכנסת תלות ב-`scipy`.
* **`top50_overlap`** - שיעור התחנות מבין `TOP_OVERLAP_N` המובילות של חלון A המופיעות גם ב-top-`N`
  של חלון B. זהו המספר החשוב מבחינה תפעולית: רשות המגנה על 50 התחנות הקריטיות ביותר רוצה לדעת
  אם הרשימה הזו משתנה בלילה.
* **`n_common` / `n_only_a` / `n_only_b`** - כמה תחנות משותפות לשני החלונות, וכמה קיימות רק
  באחד מהם.

שתי הסתייגויות כנות, המוצגות לפני שהמספרים מופיעים:

1. Spearman על התחנות ה*משותפות* מתעלם במכוון מהתחנות הנעלמות - ותחנה שפשוט אין בה שירות
   ב-02:00 היא ללא ספק הצורה החזקה ביותר של "הקריטיות השתנתה עם הזמן". אפקט זה נמדד בנפרד,
   באמצעות `n_only_a` / `n_only_b` ובאמצעות טבלת ההיעלמות בסעיף 11.
2. כל החלונות הם תת-גרפים של אותה רשת פיזית, ולכן ה*טופולוגיה* משותפת ברובה ומתאמים גבוהים
   צפויים כמעט מעצם ההגדרה. האות המעניין אינו "האם rho גבוה" - הוא יהיה - אלא **היכן rho יורד
   מתחת לרצפת הרעש של האומד**, אותה מודד סעיף 10.

In [ ]:
def spearman(a, b):
    """Spearman rho = Pearson correlation of average ranks. No scipy needed."""
    ra = pd.Series(a).rank(method='average').to_numpy(dtype=float)
    rb = pd.Series(b).rank(method='average').to_numpy(dtype=float)
    if len(ra) < 3 or ra.std() == 0 or rb.std() == 0:
        return float('nan')
    return float(np.corrcoef(ra, rb)[0, 1])


def top_set(df, metric, n=TOP_OVERLAP_N):
    return set(df.nsmallest(n, 'rank_' + metric)['stop_id'])


agree_rows = []
if len(WINDOW_NAMES) >= 2:
    for i, a in enumerate(WINDOW_NAMES):
        for b in WINDOW_NAMES[i + 1:]:
            da, db = metric_frames[a], metric_frames[b]
            sa, sb = set(da['stop_id']), set(db['stop_id'])
            common = sorted(sa & sb)
            ma = da.set_index('stop_id').loc[common]
            mb = db.set_index('stop_id').loc[common]
            for metric in AGREEMENT_METRICS:
                ta, tb = top_set(da, metric), top_set(db, metric)
                agree_rows.append({
                    'window_a': a, 'window_b': b, 'metric': metric,
                    'n_common': len(common), 'n_only_a': len(sa - sb), 'n_only_b': len(sb - sa),
                    'spearman_rho': round(spearman(ma[metric], mb[metric]), 4),
                    'top' + str(TOP_OVERLAP_N) + '_overlap':
                        round(len(ta & tb) / float(TOP_OVERLAP_N), 4),
                })

window_agreement = pd.DataFrame(agree_rows)
if len(window_agreement):
    window_agreement.to_csv(TABLES / 'window_rank_agreement.csv', index=False, encoding='utf-8-sig')
    print(window_agreement.to_string(index=False))
else:
    print('fewer than two windows - no cross-window agreement to compute.')

### 9ב. מפות חום של הסכמה

אותם מספרים כתמונה: מפת חום אחת לכל מידה המחזיקה את מתאמי ה-Spearman הזוגיים, ואחת המחזיקה את
חפיפות ה-top-50 עבור `RANK_METRIC`. האלכסון הוא 1 מעצם ההגדרה ומצויר אך ורק כנקודת ייחוס.
התאים מסומנים בערכים משום שעם מספר מועט של חלונות הערך המדויק חשוב יותר מהצבע.

In [ ]:
def pair_matrix(df, metric, value_col, diag=1.0):
    M = pd.DataFrame(np.full((len(WINDOW_NAMES), len(WINDOW_NAMES)), np.nan),
                     index=WINDOW_NAMES, columns=WINDOW_NAMES, dtype=float)
    np.fill_diagonal(M.values, diag)
    sub = df[df['metric'] == metric]
    for _, r in sub.iterrows():
        M.loc[r['window_a'], r['window_b']] = r[value_col]
        M.loc[r['window_b'], r['window_a']] = r[value_col]
    return M


if len(window_agreement):
    ov_col = 'top' + str(TOP_OVERLAP_N) + '_overlap'
    panels = [(m, 'spearman_rho', 'Spearman rho - ' + m) for m in AGREEMENT_METRICS]
    panels.append((RANK_METRIC, ov_col, 'top-' + str(TOP_OVERLAP_N) + ' overlap - ' + RANK_METRIC))

    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.0))
    axes = np.atleast_1d(axes)
    for ax, (metric, col, title) in zip(axes, panels):
        M = pair_matrix(window_agreement, metric, col)
        im = ax.imshow(M.values, vmin=0, vmax=1, cmap='viridis')
        ax.set_xticks(range(len(WINDOW_NAMES)))
        ax.set_xticklabels(WINDOW_NAMES, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(WINDOW_NAMES)))
        ax.set_yticklabels(WINDOW_NAMES, fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.grid(False)
        for i in range(len(WINDOW_NAMES)):
            for j in range(len(WINDOW_NAMES)):
                v = M.values[i, j]
                if np.isfinite(v):
                    ax.text(j, i, format(v, '.2f'), ha='center', va='center', fontsize=8,
                            color='white' if v < 0.6 else 'black')
    fig.colorbar(im, ax=list(axes), fraction=0.02, pad=0.02)
    fig.suptitle('Do the station rankings agree between time windows?', fontsize=13)
    fig.savefig(FIGURES / 'rank_agreement_heatmap.png', dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print('saved', FIGURES / 'rank_agreement_heatmap.png')
else:
    print('skipped - needs at least two windows.')

## 10. רצפת הרעש של עדשת ה-betweenness

`approx_betweenness` הוא מדגם של 250 pivots של גודל המוגדר על פני כ-25,000 מקורות. שני מדגמים
בלתי תלויים של **אותו** גרף לא יפיקו את אותו דירוג, ולכן חלק מכל אי-הסכמה בין חלונות הוא האומד
המדבר, ולא הרשת. כדי להפריד בין שני האפקטים אנו מריצים שוב betweenness על חלון השיא עם seed
שני (`BETWEENNESS_SEED_B`) ומודדים את מתאם ה-Spearman ואת חפיפת ה-top-50 בין שני המדגמים.

יש לקרוא את התוצאה כ**תקרה**: הסכמה בין חלונות ברמה זו או מעליה אינה ניתנת להבחנה מ"אותו דירוג
שנמדד פעמיים", ורק הסכמה הנמוכה ממנה באופן ברור מהווה ראיה לכך שהקריטיות אכן נעה עם שעת היום.
הגדירו `RUN_NOISE_FLOOR = False` כדי לדלג על הרצת ה-betweenness הנוספת.

In [ ]:
noise_rows = []
if RUN_NOISE_FLOOR and RANK_METRIC == 'approx_betweenness' and PEAK_WINDOW is not None:
    print('re-running betweenness on window "' + str(PEAK_WINDOW) + '" with seed',
          BETWEENNESS_SEED_B, '...')
    alt = window_metrics(PEAK_WINDOW, GRAPHS[PEAK_WINDOW], seed=BETWEENNESS_SEED_B)
    base = metric_frames[PEAK_WINDOW]
    j = base[['stop_id', 'approx_betweenness', 'rank_approx_betweenness']].merge(
        alt[['stop_id', 'approx_betweenness', 'rank_approx_betweenness']],
        on='stop_id', suffixes=('_a', '_b'))
    ta = set(base.nsmallest(TOP_OVERLAP_N, 'rank_approx_betweenness')['stop_id'])
    tb = set(alt.nsmallest(TOP_OVERLAP_N, 'rank_approx_betweenness')['stop_id'])
    noise_rows.append({
        'window': PEAK_WINDOW, 'seed_a': BETWEENNESS_SEED, 'seed_b': BETWEENNESS_SEED_B,
        'k_pivots': BETWEENNESS_SAMPLES, 'n_stations': len(j),
        'spearman_rho_same_window': round(spearman(j['approx_betweenness_a'],
                                                   j['approx_betweenness_b']), 4),
        'top' + str(TOP_OVERLAP_N) + '_overlap_same_window':
            round(len(ta & tb) / float(TOP_OVERLAP_N), 4),
        'nonzero_share_seed_a': round(float((j['approx_betweenness_a'] > 0).mean()), 4),
    })
    noise_floor = pd.DataFrame(noise_rows)
    noise_floor.to_csv(TABLES / 'betweenness_noise_floor.csv', index=False, encoding='utf-8-sig')
    print()
    print(noise_floor.to_string(index=False))
    print()
    print('Any cross-window Spearman above',
          noise_floor.iloc[0]['spearman_rho_same_window'],
          'is within the sampling noise of the estimator and must NOT be read as "the ranking is stable in time".')
else:
    noise_floor = pd.DataFrame()
    print('noise-floor check skipped (RUN_NOISE_FLOOR =', RUN_NOISE_FLOOR,
          ', RANK_METRIC =', RANK_METRIC, ')')

## 11. אילו תחנות זזות הכי הרבה בין שעת שיא לשעת שפל?

ההשוואה הישירה. שתי תופעות שונות מופרדות במכוון:

* **מזיזים (Movers)** - תחנות הקיימות ב*שני* החלונות אשר ה-`rank_in_window` שלהן משתנה. כדי לשמור
  על משמעות אנו שוקלים רק תחנות הנמצאות בתוך `MOVER_POOL` המובילות של לפחות אחד משני החלונות:
  תחנה הנעה מדירוג 18,400 לדירוג 21,900 היא רעש זנב, לא ממצא. `rank_shift = rank_offpeak - rank_peak`,
  כך שהיסט **חיובי** משמעו שהתחנה *פחות* מרכזית בשעת שפל (היא חשובה בעיקר בשעת העומס) והיסט
  **שלילי** משמעו שהיא *מרכזית יחסית יותר* כאשר השירות מידלדל - בדרך כלל תחנה על מסדרון הממשיך
  לפעול כל הלילה בעוד ששכנותיה נעצרות.
* **תחנות שנעלמו** - תחנות המשורתות בחלון השיא ללא **שום שירות כלל** בחלון השפל. לאלה אין דירוג
  להשוואה; הן הראיה הנקייה ביותר לכך שהקריטיות תלוית זמן, והן מיוצאות בנפרד אל
  `tables/vanished_at_offpeak.csv`, ממוינות לפי מידת מרכזיותן בשעת השיא.

שתי הטבלאות נושאות שמות, אזור וקואורדינטות כך שניתן לאתרן על מפה.

In [ ]:
movers = pd.DataFrame()
vanished = pd.DataFrame()

if OFFPEAK_WINDOW is not None:
    pk = metric_frames[PEAK_WINDOW].set_index('stop_id')
    op = metric_frames[OFFPEAK_WINDOW].set_index('stop_id')
    common = sorted(set(pk.index) & set(op.index))

    movers = pd.DataFrame({'stop_id': common})
    movers['stop_name'] = movers['stop_id'].map(pk['stop_name'])
    movers['region'] = movers['stop_id'].map(pk['region'])
    movers['lat'] = movers['stop_id'].map(pk['lat'])
    movers['lon'] = movers['stop_id'].map(pk['lon'])
    movers['metric'] = RANK_METRIC
    movers['rank_peak'] = movers['stop_id'].map(pk['rank_in_window']).astype(int)
    movers['rank_offpeak'] = movers['stop_id'].map(op['rank_in_window']).astype(int)
    movers['peak_window'] = PEAK_WINDOW
    movers['offpeak_window'] = OFFPEAK_WINDOW
    movers['degree_peak'] = movers['stop_id'].map(pk['degree']).astype(float)
    movers['degree_offpeak'] = movers['stop_id'].map(op['degree']).astype(float)
    movers['weighted_degree_peak'] = movers['stop_id'].map(pk['weighted_degree']).astype(float)
    movers['weighted_degree_offpeak'] = movers['stop_id'].map(op['weighted_degree']).astype(float)
    movers['rank_shift'] = movers['rank_offpeak'] - movers['rank_peak']
    movers['abs_rank_shift'] = movers['rank_shift'].abs()
    movers['in_pool'] = (movers[['rank_peak', 'rank_offpeak']].min(axis=1) <= MOVER_POOL)
    movers['direction'] = np.where(movers['rank_shift'] > 0, 'less central off-peak',
                          np.where(movers['rank_shift'] < 0, 'more central off-peak', 'unchanged'))
    movers = movers.sort_values('abs_rank_shift', ascending=False).reset_index(drop=True)
    movers.to_csv(TABLES / 'rank_movers.csv', index=False, encoding='utf-8-sig')

    pool = movers[movers['in_pool']]
    print('stations in both windows :', format(len(movers), ','),
          '| inside the top-' + str(MOVER_POOL), 'pool of either window:', format(len(pool), ','))
    print()
    print('--- biggest DROP off-peak (critical at rush hour only) ---')
    print(pool.nlargest(10, 'rank_shift')[
        ['stop_name', 'region', 'rank_peak', 'rank_offpeak', 'rank_shift']].to_string(index=False))
    print()
    print('--- biggest RISE off-peak (relatively more critical when service thins) ---')
    print(pool.nsmallest(10, 'rank_shift')[
        ['stop_name', 'region', 'rank_peak', 'rank_offpeak', 'rank_shift']].to_string(index=False))

    # --- stations that simply have no service in the off-peak window ---
    gone = sorted(set(pk.index) - set(op.index))
    vanished = pk.loc[gone, ['stop_name', 'region', 'lat', 'lon', 'degree',
                             'weighted_degree', 'approx_betweenness', 'rank_in_window']].copy()
    vanished = (vanished.rename(columns={'rank_in_window': 'rank_in_peak_window'})
                        .reset_index()
                        .sort_values('rank_in_peak_window'))
    vanished['peak_window'] = PEAK_WINDOW
    vanished['offpeak_window'] = OFFPEAK_WINDOW
    vanished.to_csv(TABLES / 'vanished_at_offpeak.csv', index=False, encoding='utf-8-sig')
    print()
    print('stations served in "' + str(PEAK_WINDOW) + '" but with NO service in "' +
          str(OFFPEAK_WINDOW) + '":', format(len(vanished), ','),
          '(' + str(round(100 * len(vanished) / max(len(pk), 1), 1)) + '% of the peak network)')
    if len(vanished):
        print(vanished.head(10)[['stop_name', 'region', 'rank_in_peak_window',
                                 'degree', 'weighted_degree']].to_string(index=False))
else:
    print('skipped - needs two windows.')

### 11ב. מזיזי הדירוג, בתרשים

תרשים עמודות אופקי יחיד של `TOP_MOVERS` ההיסטים הגדולים ביותר בכל כיוון, תוך שימוש בשמות
התחנות בעברית (המעובדים דרך תיקון ה-bidi מסעיף 3). עמודות המצביעות ימינה הן תחנות המאבדות
דירוג בשעת שפל; עמודות המצביעות שמאלה הן תחנות המרוויחות דירוג. זוג הדירוגים מודפס בקצה כל
עמודה כך שניתן לשפוט את הסדר גודל, ולא רק את הכיוון.

In [ ]:
if len(movers):
    pool = movers[movers['in_pool']]
    down = pool.nlargest(TOP_MOVERS, 'rank_shift')
    up = pool.nsmallest(TOP_MOVERS, 'rank_shift')
    sel = pd.concat([up, down]).drop_duplicates('stop_id').sort_values('rank_shift')

    labels = [(nm if isinstance(nm, str) and nm else sid) + '  (' + str(rp) + '->' + str(ro) + ')'
              for nm, sid, rp, ro in zip(sel['stop_name'], sel['stop_id'],
                                         sel['rank_peak'], sel['rank_offpeak'])]
    colors = ['#2563eb' if v < 0 else '#dc2626' for v in sel['rank_shift']]

    fig, ax = plt.subplots(figsize=(11, max(5, 0.32 * len(sel))))
    ax.barh(range(len(sel)), sel['rank_shift'], color=colors, alpha=0.9)
    ax.set_yticks(range(len(sel)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color='#374151', linewidth=1)
    ax.set_xlabel('rank shift  =  rank in "' + str(OFFPEAK_WINDOW) + '"  -  rank in "' +
                  str(PEAK_WINDOW) + '"   (positive = less central off-peak)')
    ax.set_title('Stations whose ' + RANK_METRIC + ' rank moves most between peak and off-peak\n'
                 '(blue = more central off-peak, red = less central off-peak; '
                 'top-' + str(MOVER_POOL) + ' pool only)', fontsize=11)
    fig.tight_layout()
    fig.savefig(FIGURES / 'top_rank_movers.png', dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print('saved', FIGURES / 'top_rank_movers.png')
else:
    print('skipped - no mover table.')

## 12. סימולציית התקיפה לפי חלון

המנוע הוא זה שאומת במחברה `06`, בתוספת שני התיקונים שהיא ביססה:

* **סדרי תקיפה נטולי כפילויות** - מיקום `k` בסדר תקיפה הוא תמיד התחנה ה-`k` ה*נבדלת*, כך שציר
  ה-x לעולם אינו מפריז במספר התחנות שנמחקו בפועל (נאכף בבדיקת assert בכל צעד);
* **נרמול לפי צמתים ששרדו** - המידה המדווחת היא `|LCC| / (N_window - k)`, שיעור התחנות
  ה*נותרות* שעדיין ניתנות להשגה הדדית. נרמול לפי המספר המקורי היה מבטיח עקומה יורדת אפילו עבור
  רשת שלמה לחלוטין, וגם היה הופך חלונות בגדלים שונים לבלתי ברי-השוואה.

דבר אחד חדש באמת כאן. החלונות מכילים מספרים שונים של תחנות, ולכן הסרה של "50 תחנות" אינה
משמעה אותו דבר בכל אחד מהם. לפיכך רשת ההסרה מוגדרת כ**שיעור ממספר התחנות של אותו חלון**
(`REMOVAL_FRACTIONS`, בתוספת עוגני k קטן מ-`EXTRA_FRACTIONS`), והעקומות משורטטות כנגד שיעור זה.
הספירה המוחלטת עדיין נרשמת בעמודה `removed`, כפי שחוזה הנתונים דורש.

מדומים ארבעה סדרי תקיפה ממוקדים (degree, weighted degree, betweenness מדגמי, ו-articulation
points תחילה) בתוספת קו בסיס אקראי הממוצע על פני `RANDOM_TRIALS` תמורות עם seed. שימוש בתמורה
אחת לכל ניסוי - במקום מדגם בלתי תלוי לכל `k` - הופך כל ניסוי לרצף כשל הדרגתי קוהרנטי עם הסרות
מקוננות, וכך נראה כשל מדורג אמיתי.

In [ ]:
def dedup_keep_order(seq):
    """Remove duplicates but keep first-occurrence order."""
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def k_grid_for(n):
    """Removal counts for a window of n stations, from the fraction grid."""
    fracs = sorted(set([f for f in REMOVAL_FRACTIONS] + [f for f in EXTRA_FRACTIONS]))
    ks = sorted({int(round(f * n)) for f in fracs if 0 <= f <= 1})
    return [k for k in ks if k < n]


def lcc_size(G, node_set, removed):
    """Size of the largest connected component after deleting `removed` (a set of node ids)."""
    survivors = node_set.difference(removed)
    if not survivors:
        return 0
    return max((len(c) for c in nx.connected_components(G.subgraph(survivors))), default=0)


def attack_orders(name, G, mdf):
    """The deterministic attack orders for one window, all duplicate-free."""
    m = mdf.copy()
    def by(col):
        s = m.sort_values([col, 'weighted_degree', 'degree', 'stop_id'],
                          ascending=[False, False, False, True])
        return s['stop_id'].tolist()

    aps = sorted(set(nx.articulation_points(G)))
    ap_rank = m.set_index('stop_id')['degree'].to_dict()
    aps = sorted(aps, key=lambda n: (-float(ap_rank.get(n, 0)), n))

    orders = {
        'degree': by('degree'),
        'weighted degree': by('weighted_degree'),
        'betweenness': by('approx_betweenness'),
        # articulation points first (ordered by degree), then everything else by degree
        'articulation points': dedup_keep_order(aps + by('degree')),
    }
    orders = {k: v for k, v in orders.items() if k in ATTACK_STRATEGIES}
    for k, v in orders.items():
        assert len(v) == len(set(v)), 'duplicate entries in strategy ' + k + ' / window ' + str(name)
    return orders, len(aps)


def simulate_window(name, G, mdf):
    """Run every strategy + the random baseline inside one window."""
    n0 = G.number_of_nodes()
    node_set = set(G.nodes())
    ks = k_grid_for(n0)
    orders, n_aps = attack_orders(name, G, mdf)
    rows = []

    def _row(strategy, k, size):
        surviving = n0 - k
        return {'window': name, 'strategy': strategy, 'removed': k,
                'removed_fraction': round(k / n0, 5) if n0 else 0.0,
                'window_nodes': n0, 'surviving_nodes': surviving, 'lcc_size': float(size),
                'largest_component_share_surviving': round(size / surviving, 5) if surviving > 0 else 0.0,
                'lcc_share_original': round(size / n0, 5) if n0 else 0.0}

    t0 = time.time()
    for strategy, order in orders.items():
        order = [x for x in order if x in node_set]
        for k in ks:
            removed = set(order[:k])
            assert len(removed) == k, 'strategy ' + strategy + ' removed ' + str(len(removed)) + ' != ' + str(k)
            rows.append(_row(strategy, k, lcc_size(G, node_set, removed)))

    sizes = {k: [] for k in ks}
    all_nodes = list(node_set)
    for t in range(RANDOM_TRIALS):
        rng = random.Random(SEED + t)
        perm = list(all_nodes)
        rng.shuffle(perm)
        for k in ks:
            sizes[k].append(lcc_size(G, node_set, set(perm[:k])))
    for k in ks:
        rows.append(_row('random (baseline)', k, float(np.mean(sizes[k]))))

    df = pd.DataFrame(rows)
    end = df[df['removed'] == max(ks)]
    print('  ' + str(name).ljust(18), format(n0, '>7,'), 'nodes |', len(ks), 'removal levels |',
          format(n_aps, ','), 'articulation points |', str(round(time.time() - t0, 1)) + 's')
    for _, r in end.sort_values('largest_component_share_surviving').iterrows():
        print('      at ' + str(round(100 * r['removed_fraction'], 1)) + '% removed  ' +
              str(r['strategy']).ljust(20) + ' LCC/surviving = ' +
              format(r['largest_component_share_surviving'], '.3f'))
    return df


print('running the attack simulation per window (this is the expensive cell) ...')
t_all = time.time()
sim_frames = [simulate_window(name, GRAPHS[name], metric_frames[name]) for name in WINDOW_NAMES]
resilience_detail = pd.concat(sim_frames, ignore_index=True)
print()
print('total simulation time:', round(time.time() - t_all, 1), 's |',
      format(len(resilience_detail), ','), 'rows')

### 12ב. כתיבת טבלאות העמידות

`tables/dynamic_resilience.csv` היא טבלת החוזה ומחזיקה בדיוק את ארבע העמודות המוסכמות
(`window, strategy, removed, largest_component_share_surviving`). כל שאר מה שהסימולציה רשמה -
שיעור ההסרה, גודל החלון, מספר השורדים, גודל ה-LCC הגולמי והנרמול הישן `|LCC| / N_original`
שנשמר אך ורק לצורך השוואתיות עם מחברה 06 - עובר אל `tables/dynamic_resilience_detail.csv`, כך
שצרכנים במורד הזרם הקוראים את טבלת החוזה מקבלים בדיוק את מה שהובטח להם.

In [ ]:
CONTRACT = ['window', 'strategy', 'removed', 'largest_component_share_surviving']
resilience_detail.to_csv(TABLES / 'dynamic_resilience_detail.csv', index=False, encoding='utf-8-sig')
resilience_detail[CONTRACT].to_csv(TABLES / 'dynamic_resilience.csv', index=False, encoding='utf-8-sig')
print('wrote', TABLES / 'dynamic_resilience.csv', '(' + format(len(resilience_detail), ','), 'rows )')
print(resilience_detail[CONTRACT].head(6).to_string(index=False))

## 13. עקומות נזק: האם הרשת שבירה יותר כאשר השירות דליל?

איור הכותרת. שני הפאנלים משרטטים את `|LCC| / (N - k)` כנגד ה**שיעור** מתחנות החלון שהוסרו, כך
שחלונות בגדלים שונים יכולים להיפרש על אותם צירים.

* **שמאל - תקיפה ממוקדת (סדר לפי degree).** קו אחד לכל חלון. אם עקומת הלילה יורדת מהר יותר
  מעקומת השיא, אזי רשת דלילה אכן קלה יותר לניפוץ: עם פחות שירותים מקבילים יש פחות מסלולים
  חלופיים, ולכן כל תחנה שמוסרת לוקחת עמה יותר קישוריות.
* **ימין - כשל אקראי.** הביקורת (control). הסרה אקראית מתעלמת ממבנה, ולכן כל הפרדה בין חלונות
  כאן משקפת את הדלילות הגולמית של גרף החלון ולא את תחכומו של התוקף. המרחק בין הפאנל השמאלי
  לימני *עבור אותו חלון* הוא החלק מהנזק המיוחס למיקוד.

עקומה הנשארת קרוב ל-1.0 משמעה שהרשת רק התכווצה; עקומה הצוללת משמעה שהרשת התפרקה.

In [ ]:
WCOLORS = plt.cm.viridis(np.linspace(0.08, 0.88, max(len(WINDOW_NAMES), 1)))
WCOLOR = {w: WCOLORS[i] for i, w in enumerate(WINDOW_NAMES)}

panels = [('degree', 'Targeted attack - highest degree first'),
          ('random (baseline)', 'Random failure (baseline, ' + str(RANDOM_TRIALS) + ' trials averaged)')]
panels = [(s, t) for s, t in panels if s in set(resilience_detail['strategy'])]

fig, axes = plt.subplots(1, len(panels), figsize=(7.0 * len(panels), 5.4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (strategy, title) in zip(axes, panels):
    sub = resilience_detail[resilience_detail['strategy'] == strategy]
    for w in WINDOW_NAMES:
        g = sub[sub['window'] == w].sort_values('removed_fraction')
        if g.empty:
            continue
        n0 = int(g['window_nodes'].iloc[0])
        ax.plot(100 * g['removed_fraction'], g['largest_component_share_surviving'],
                marker='o', markersize=3.0, linewidth=1.9, color=WCOLOR[w],
                label=str(w) + '  (' + format(n0, ',') + ' stations)')
    ax.set_xlabel('stations removed (% of that window)')
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, 1.03)
axes[0].set_ylabel('|LCC| / surviving stations')
axes[0].legend(loc='lower left', fontsize=8, framealpha=0.9)
fig.suptitle('Dynamic resilience: damage curves per time window', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES / 'damage_curves_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('saved', FIGURES / 'damage_curves_by_window.png')

### 13ב. כל אסטרטגיה, חלון אחר חלון

אותה סימולציה כ-small multiples - פאנל אחד לכל חלון, כל אסטרטגיות התקיפה בתוספת קו הבסיס
האקראי. זו התצוגה העונה על "איזו תפיסת חשיבות בוחרת את המטרות ההרסניות ביותר *בשעה זו*": אם
סדר האסטרטגיות משתנה בין פאנלים, אזי אפילו אסטרטגיית התקיפה הטובה ביותר היא תלוית זמן.

In [ ]:
strats = [s for s in ATTACK_STRATEGIES if s in set(resilience_detail['strategy'])] + ['random (baseline)']
SCOLORS = {'degree': '#dc2626', 'weighted degree': '#ea580c', 'betweenness': '#2563eb',
           'articulation points': '#7c3aed', 'random (baseline)': '#6b7280'}

ncol = min(3, len(WINDOW_NAMES))
nrow = int(np.ceil(len(WINDOW_NAMES) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 4.4 * nrow), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()
for ax, w in zip(axes, WINDOW_NAMES):
    sub = resilience_detail[resilience_detail['window'] == w]
    for s in strats:
        g = sub[sub['strategy'] == s].sort_values('removed_fraction')
        if g.empty:
            continue
        ax.plot(100 * g['removed_fraction'], g['largest_component_share_surviving'],
                linewidth=1.7, color=SCOLORS.get(s, '#111827'),
                linestyle='--' if s == 'random (baseline)' else '-', label=s)
    n0 = int(sub['window_nodes'].iloc[0]) if len(sub) else 0
    ax.set_title(str(w) + '  (' + format(n0, ',') + ' stations)', fontsize=10)
    ax.set_ylim(0, 1.03)
for ax in axes[len(WINDOW_NAMES):]:
    ax.axis('off')
axes[0].legend(fontsize=8, loc='lower left')
for ax in axes[:len(WINDOW_NAMES)]:
    ax.set_xlabel('stations removed (%)')
    ax.set_ylabel('|LCC| / surviving')
fig.suptitle('Attack strategies inside each time window', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES / 'damage_curves_grid.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('saved', FIGURES / 'damage_curves_grid.png')

## 14. ניקוד העקומות

עקומות מראות צורה; הדוח זקוק למספרים. עבור כל חלון x אסטרטגיה אנו מחשבים:

* **`auc`** - השטח מתחת לעקומת `|LCC| / (N - k)` על פני ציר שיעור ההסרה, מנורמל מחדש ל-`[0, 1]`
  על ידי חלוקה ברוחב הציר. רשת עמידה לחלוטין מקבלת ציון של כ-1; רשת המתנפצת מיד מקבלת ציון של
  כ-0. מכיוון שציר ה-x הוא *שיעור* מכל חלון, ערכי ה-AUC של חלונות שונים ברי-השוואה ישירה.
* **`share_at_1pct` / `share_at_5pct` / `share_at_10pct`** - שיעור ה-LCC המנורמל לפי שורדים לאחר
  הסרת 1%, 5% ו-10% מתחנות אותו חלון (נקודת הרשת הקרובה ביותר).
* **`targeting_gain`** - `auc(random) - auc(strategy)`: כמה נזק נוסף האסטרטגיה גורמת בהשוואה
  למזל רע. זהו המספר האומר אם *מיקוד* חשוב יותר בשעות מסוימות מאשר באחרות.

In [ ]:
def _at(g, frac):
    if g.empty:
        return float('nan')
    i = (g['removed_fraction'] - frac).abs().idxmin()
    return float(g.loc[i, 'largest_component_share_surviving'])


rows = []
for w in WINDOW_NAMES:
    sub = resilience_detail[resilience_detail['window'] == w]
    aucs = {}
    for s in sub['strategy'].unique():
        g = sub[sub['strategy'] == s].sort_values('removed_fraction')
        x = g['removed_fraction'].to_numpy(dtype=float)
        y = g['largest_component_share_surviving'].to_numpy(dtype=float)
        span = x.max() - x.min()
        aucs[s] = float(_trapz(y, x) / span) if span > 0 else float('nan')
    base = aucs.get('random (baseline)', float('nan'))
    for s in sub['strategy'].unique():
        g = sub[sub['strategy'] == s].sort_values('removed_fraction')
        rows.append({'window': w, 'strategy': s,
                     'window_nodes': int(sub['window_nodes'].iloc[0]),
                     'auc': round(aucs[s], 4),
                     'targeting_gain_vs_random': round(base - aucs[s], 4),
                     'share_at_1pct': round(_at(g, 0.01), 4),
                     'share_at_5pct': round(_at(g, 0.05), 4),
                     'share_at_10pct': round(_at(g, 0.10), 4)})

fragility = pd.DataFrame(rows).sort_values(['window', 'auc'])
fragility.to_csv(TABLES / 'fragility_summary.csv', index=False, encoding='utf-8-sig')
print(fragility.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
piv = fragility.pivot(index='window', columns='strategy', values='auc').reindex(WINDOW_NAMES)
piv.plot(kind='bar', ax=axes[0], color=[SCOLORS.get(c, '#111827') for c in piv.columns], width=0.82)
axes[0].set_ylabel('AUC of |LCC| / surviving  (1 = perfectly robust)')
axes[0].set_title('Robustness score per window and strategy', fontsize=11)
axes[0].set_ylim(0, 1.02)
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=25)

piv2 = fragility.pivot(index='window', columns='strategy',
                       values='share_at_5pct').reindex(WINDOW_NAMES)
piv2.plot(kind='bar', ax=axes[1], color=[SCOLORS.get(c, '#111827') for c in piv2.columns], width=0.82)
axes[1].set_ylabel('|LCC| / surviving after removing 5% of stations')
axes[1].set_title('Damage after a 5% attack', fontsize=11)
axes[1].set_ylim(0, 1.02)
axes[1].get_legend().remove()
axes[1].tick_params(axis='x', rotation=25)

fig.suptitle('Is the network more fragile when service is thin?', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES / 'fragility_summary.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('saved', FIGURES / 'fragility_summary.png')

## 15. סיכום קריא-מכונה

כל מה שמחברת מאוחרת יותר (או הדוח) עשויים לרצות לצטט, בקובץ JSON יחיד: החלונות שנותחו וגדליהם,
איזה חלון טופל כשיא ומדוע, טווח ההסכמה בין החלונות, רצפת הרעש של ה-betweenness, ציוני השבירות,
כמה תחנות נעלמות בשעת שפל, וכל קבוע שהיה בתוקף. כתיבת הקבועים אל תוך הסיכום משמעה שמספר
המצוטט בדוח תמיד ניתן למעקב לאחור אל ההגדרות שהפיקו אותו.

In [ ]:
def _f(x):
    try:
        v = float(x)
        return None if not np.isfinite(v) else round(v, 6)
    except (TypeError, ValueError):
        return None


def _json_default(o):
    """numpy scalars and Paths are not JSON-serialisable; coerce them."""
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, Path):
        return str(o)
    return str(o)


summary = {
    'stage': '20_dynamic_resilience',
    'source_stage': str(STAGE19.name),
    'windows': [{'window': r['window'], 'start_hour': (None if pd.isna(r['start_hour']) else r['start_hour']),
                 'end_hour': (None if pd.isna(r['end_hour']) else r['end_hour']),
                 'nodes': int(r['nodes']), 'undirected_edges': int(r['undirected_edges']),
                 'avg_degree': _f(r['avg_degree']),
                 'largest_component_share': _f(r['largest_component_share'])}
                for _, r in window_graph_summary.iterrows()],
    'peak_window': PEAK_WINDOW, 'peak_window_rule': WHY_PEAK,
    'offpeak_window': OFFPEAK_WINDOW, 'offpeak_window_rule': WHY_OFFPEAK,
    'rank_metric': RANK_METRIC,
    'cross_window_agreement': (
        {} if not len(window_agreement) else {
            m: {'spearman_min': _f(window_agreement.loc[window_agreement['metric'] == m, 'spearman_rho'].min()),
                'spearman_max': _f(window_agreement.loc[window_agreement['metric'] == m, 'spearman_rho'].max()),
                'top_overlap_min': _f(window_agreement.loc[window_agreement['metric'] == m,
                                                           'top' + str(TOP_OVERLAP_N) + '_overlap'].min()),
                'top_overlap_max': _f(window_agreement.loc[window_agreement['metric'] == m,
                                                           'top' + str(TOP_OVERLAP_N) + '_overlap'].max())}
            for m in AGREEMENT_METRICS}),
    'betweenness_noise_floor': ({} if not len(noise_floor)
                                else {k: (_f(v) if isinstance(v, (int, float, np.floating)) else v)
                                      for k, v in noise_floor.iloc[0].to_dict().items()}),
    'movers': ({} if not len(movers) else {
        'stations_in_both_windows': int(len(movers)),
        'pool_size': int(movers['in_pool'].sum()),
        'max_abs_rank_shift_in_pool': int(movers.loc[movers['in_pool'], 'abs_rank_shift'].max())
                                      if movers['in_pool'].any() else 0,
        'median_abs_rank_shift_in_pool': _f(movers.loc[movers['in_pool'], 'abs_rank_shift'].median())
                                         if movers['in_pool'].any() else None,
        'vanished_at_offpeak': int(len(vanished))}),
    'fragility': [{'window': r['window'], 'strategy': r['strategy'], 'auc': _f(r['auc']),
                   'share_at_5pct': _f(r['share_at_5pct']),
                   'targeting_gain_vs_random': _f(r['targeting_gain_vs_random'])}
                  for _, r in fragility.iterrows()],
    'constants': {'BETWEENNESS_SAMPLES': BETWEENNESS_SAMPLES, 'BETWEENNESS_SEED': BETWEENNESS_SEED,
                  'BETWEENNESS_SEED_B': BETWEENNESS_SEED_B, 'RUN_NOISE_FLOOR': RUN_NOISE_FLOOR,
                  'RANK_METRIC': RANK_METRIC, 'TOP_OVERLAP_N': TOP_OVERLAP_N,
                  'MOVER_POOL': MOVER_POOL, 'RANDOM_TRIALS': RANDOM_TRIALS, 'SEED': SEED,
                  'ATTACK_STRATEGIES': ATTACK_STRATEGIES,
                  'REMOVAL_FRACTIONS': [float(f) for f in REMOVAL_FRACTIONS],
                  'EXTRA_FRACTIONS': [float(f) for f in EXTRA_FRACTIONS]},
}

with open(STAGE / 'dynamic_summary.json', 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2, default=_json_default)
print('wrote', STAGE / 'dynamic_summary.json')
print(json.dumps({k: summary[k] for k in ['peak_window', 'offpeak_window', 'rank_metric',
                                          'cross_window_agreement', 'betweenness_noise_floor',
                                          'movers']},
                 ensure_ascii=False, indent=2, default=_json_default)[:1800])

## 16. מה שלב זה הפיק

רשימה סופית של כל קובץ שנכתב, כך שניתן לבדוק את השלב במבט אחד ולוודא ששתי טבלאות החוזה נושאות
בדיוק את העמודות שהובטחו.

In [ ]:
print('files under', STAGE, ':')
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print('  ' + str(p.relative_to(STAGE)).ljust(46), format(p.stat().st_size, '>10,'), 'bytes')

print()
for fname, cols in [('dynamic_resilience.csv', CONTRACT),
                    ('peak_vs_offpeak_centrality.csv', CONTRACT_COLS)]:
    got = list(pd.read_csv(TABLES / fname, nrows=1, encoding='utf-8-sig').columns)
    print(fname, '->', 'OK' if got == cols else 'MISMATCH', '|', got)

## 17. מסקנות

*(המספרים המדויקים תלויים בהגדרות החלונות שמחברה 19 בחרה ובזרעי הדגימה של ה-betweenness -
יש לקרוא אותם יחד עם `tables/` ו-`dynamic_summary.json`.)*

1. **התשובה הכנה לשאלה "האם הקריטיות תלויה בשעת היום" היא ברובה *לא*, עם חריג אחד גדול.**
   כל גרפי החלונות הם תת-גרפים של אותה רשת פיזית: אותם כבישים, אותם מסדרונות, אותן תחנות. לכן
   דירוגי degree ו-weighted degree מתואמים מאוד בין חלונות כמעט מעצם ההגדרה, ורשימות ה-top-50
   חופפות במידה רבה. זו אינה תגלית, זו תוצאה של המודל, ויש לדווח עליה ככזו במקום להלביש אותה
   כיציבות.

2. **החריג הוא שירות הנעלם כליל.** `tables/vanished_at_offpeak.csv` מפרט תחנות הנושאות שירות
   בשעת שיא ו*כלל לא* בחלון השפל. עבור תחנות אלה הקריטיות אינה זזה, היא חדלה להתקיים - ונוסע
   התקוע שם ב-01:00 הוא כשל נגישות ממשי שהגרף הסטטי בן 24 השעות של מחברות 02-06 אינו מסוגל
   לבטא כלל. זו, ולא מתאמי הדירוג, הראיה החזקה ביותר לכך שגרף ממוצע-על-פני-זמן מחמיא לרשת.

3. **לעדשת ה-betweenness יש רצפת רעש, והיא אינה קטנה.** סעיף 10 דוגם מחדש betweenness על אותו
   חלון עם seed שונה. כל מתאם Spearman בין חלונות שנצפה חייב להיות מושווה למספר זה: הסכמה מעל
   הרצפה משמעה "איננו יכולים להבחין בין הדירוגים הללו", ולא "הדירוג יציב". כל תזוזת דירוג הקטנה
   מהתזוזה בין seed ל-seed היא ארטיפקט של האומד. זו הסיבה שטבלת המזיזים מוגבלת ל-`MOVER_POOL`
   התחנות המובילות, שם ההערכה מפוענחת בצורה הטובה ביותר - וגם שם יש להתייחס לשורות בודדות
   כמועמדות לבדיקה, ולא כממצאים.

4. **שבירות עוסקת בדלילות, ולא בשעה כשלעצמה.** עקומות הנזק מנורמלות לפי תחנות ששרדו ומשורטטות
   כנגד ה*שיעור* מהחלון שהוסר, ולכן גודל החלון מנוטרל. כל הפרדה שנותרת בין חלונות משקפת כמה
   יתירות יש לרשת באותה שעה: פחות שירותים מקבילים משמעו פחות מסלולים חלופיים, ולכן כל תחנה
   שמוסרת נושאת עמה יותר קישוריות. יש לקרוא את `targeting_gain_vs_random` ב-`fragility_summary.csv`
   לפני הסקת מסקנה כלשהי לגבי תוקף - אם הרווח מצטמצם בשעת שפל, הרשת הדלילה שבירה בפני *הכל*,
   ולא ספציפית בפני מיקוד.

5. **מגבלות הניתוח הזה, נאמרות במפורש.** (א) החלונות נחתכים לפי זמן היציאה המתוכנן, ולכן נסיעה
   החוצה גבול תורמת לחלון שאליו מחברה 19 שייכה אותה; חלונות קצרים מושפעים מבחירה זו יותר
   מחלונות ארוכים. (ב) הקשתות הן שכנות טופולוגית, לא קיבולת: שני חלונות עם אותה קבוצת קשתות
   נבדלים רק במשקל, ומידות קישוריות בלתי משוקללות אינן יכולות לראות אוטובוס הנוסע פעם בשעה
   במקום כל ארבע דקות. weighted degree הוא המידה היחידה כאן שכן רואה זאת. (ג) betweenness משתמש
   במסלולים קצרים ביותר לפי מספר קפיצות, ולא לפי זמן נסיעה - גרף זמני הנסיעה של מחברה 18 היה
   מהווה מדד מרחק נאמן יותר. (ד) הסרת תחנה ממודלת כמחיקת הצומת, כלומר ללא ניתוב מחדש וללא
   תחליף; מחברה 22 מטפלת בכך בנפרד.

6. **מה זה תורם לשאלת המחקר של הפרויקט.** הניתוח הסטטי זיהה תחנות קריטיות מתוך ממוצע של 24
   שעות. מחברת זו מראה עד כמה ניתן לסמוך על תשובה זו לאורך היום: *זהות* התחנות הקריטיות היא
   ברובה בלתי תלויה בזמן, *שבירות* הרשת סביבן אינה כזו, וקבוצת התחנות שיש בהן שירות כלשהו
   מצטמצמת באופן מהותי מחוץ לשעות השיא. תוכנית עמידות הבנויה על הגרף הממוצע היא אפוא נכונה
   בקירוב לגבי *היכן* להשקיע, ואופטימית באופן שיטתי לגבי *מתי* הרשת מסוגלת לספוג כשל.